In [ ]:
# Install required libraries
!pip install -q xarray netCDF4 pyarrow openap

# Import libraries
import pandas as pd
import numpy as np
import xarray as xr
import sqlite3
import matplotlib.pyplot as plt

from openap import FuelFlow

print("Libraries imported successfully!")

In [ ]:
import pandas as pd
import xarray as xr
import sqlite3
import numpy as np

observations = pd.read_parquet('/content/observations.parquet')
matches = pd.read_parquet('/content/matches.parquet')
flights = pd.read_parquet('/content/flights.parquet')
surface = xr.open_dataset('/content/era5t_oejn_2026-10-01_surface.nc')
pressure = xr.open_dataset('/content/era5_pressure_extended.nc')
conn = sqlite3.connect('/content/aeromux_db.sqlite')

print("All datasets loaded successfully!")
print("Observations:", observations.shape)
print("Flights:", flights.shape)
print("Surface dimensions:", surface.sizes)
print("Pressure dimensions:", pressure.sizes)

In [ ]:
# Calculate surface weather variables

surface["wind_speed"] = np.sqrt(
    surface["10m_u_component_of_wind"]**2
    + surface["10m_v_component_of_wind"]**2
)

surface["temperature_C"] = (
    surface["2m_temperature"] - 273.15
)

surface["surface_pressure_hPa"] = (
    surface["surface_pressure"] / 100
)

print("Surface weather calculations completed!")

In [ ]:
# Rename pressure-level variables

pressure = pressure.rename({
    "u": "u_component_of_wind",
    "v": "v_component_of_wind",
    "t": "temperature",
    "z": "geopotential"
})

# Calculate wind speed
pressure["wind_speed"] = np.sqrt(
    pressure["u_component_of_wind"]**2 +
    pressure["v_component_of_wind"]**2
)

# Convert temperature from Kelvin to Celsius
pressure["temperature_C"] = pressure["temperature"] - 273.15

# Convert geopotential to geopotential height in meters
pressure["geopotential_height_m"] = pressure["geopotential"] / 9.80665

print("Pressure-level variables renamed and calculated!")
print("Variables:", list(pressure.data_vars))

In [ ]:
# Step 5: Check aircraft geographic coverage

print("AIRCRAFT COORDINATES")

print(
    "Latitude:",
    observations["latitude_deg"].min(),
    "to",
    observations["latitude_deg"].max()
)

print(
    "Longitude:",
    observations["longitude_deg"].min(),
    "to",
    observations["longitude_deg"].max()
)

print("\nERA5 GEOGRAPHIC COVERAGE")

print(
    "Latitude:",
    float(surface.latitude.min()),
    "to",
    float(surface.latitude.max())
)

print(
    "Longitude:",
    float(surface.longitude.min()),
    "to",
    float(surface.longitude.max())
)

# Identify observations inside ERA5 coverage

inside = (
    observations["latitude_deg"].between(
        float(surface.latitude.min()),
        float(surface.latitude.max())
    )
    &
    observations["longitude_deg"].between(
        float(surface.longitude.min()),
        float(surface.longitude.max())
    )
)

print("\nObservations inside ERA5 coverage:", inside.sum())
print("Observations outside ERA5 coverage:", (~inside).sum())

print(
    "Coverage percentage:",
    round(inside.mean() * 100, 2),
    "%"
)

In [ ]:
import matplotlib.pyplot as plt

# Plot aircraft locations and ERA5 coverage
plt.figure(figsize=(10, 7))

# Plot all aircraft observations
plt.scatter(
    observations["longitude_deg"],
    observations["latitude_deg"],
    s=1,
    alpha=0.15,
    label="Aircraft Observations"
)

# Plot ERA5 geographic coverage
plt.plot(
    [38, 41, 41, 38, 38],
    [20, 20, 23, 23, 20],
    linewidth=2,
    label="ERA5 Coverage"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Aircraft Observations vs ERA5 Weather Coverage")

plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Step 6: Match surface weather with aircraft observations

# Sort latitude because ERA5 latitude may be descending
surface_sorted = surface.sortby("latitude")

# Prepare output columns
weather_columns = [
    "wind_u_10m",
    "wind_v_10m",
    "wind_speed_10m",
    "temperature_2m_C",
    "surface_pressure_hPa"
]

# Initialize weather columns
for col in weather_columns:
    observations[col] = np.nan

# Process observations in chunks to reduce memory usage
chunk_size = 50000

for start in range(0, len(observations), chunk_size):

    end = min(start + chunk_size, len(observations))

    chunk = observations.iloc[start:end]
    valid = inside.iloc[start:end]

    if not valid.any():
        continue

    selected = chunk.loc[valid]

    times = (
        selected["timestamp_utc"]
        .dt.tz_localize(None)
        .to_numpy()
    )

    # Interpolate weather based on time and location
    weather = surface_sorted[
        [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "wind_speed",
            "temperature_C",
            "surface_pressure_hPa"
        ]
    ].interp(
        time=xr.DataArray(times, dims="obs"),
        latitude=xr.DataArray(
            selected["latitude_deg"].to_numpy(), dims="obs"
        ),
        longitude=xr.DataArray(
            selected["longitude_deg"].to_numpy(), dims="obs"
        ),
        method="linear"
    )

    idx = selected.index

    observations.loc[idx, weather_columns] = np.column_stack([
        weather["10m_u_component_of_wind"].values,
        weather["10m_v_component_of_wind"].values,
        weather["wind_speed"].values,
        weather["temperature_C"].values,
        weather["surface_pressure_hPa"].values
    ])

    print(f"Processed {end:,} / {len(observations):,}")

print("\nSurface weather matching completed!")

In [ ]:
# Step 7: Check surface weather matching results

weather_columns = [
    "wind_u_10m",
    "wind_v_10m",
    "wind_speed_10m",
    "temperature_2m_C",
    "surface_pressure_hPa"
]

print("Total observations:", len(observations))

print("Matched observations:",
      observations["wind_speed_10m"].notna().sum())

print("Missing observations:",
      observations["wind_speed_10m"].isna().sum())

print("Coverage percentage:",
      round(observations["wind_speed_10m"].notna().mean() * 100, 2), "%")

print("\nMissing values by weather variable:")
print(observations[weather_columns].isna().sum())

In [ ]:
# Step 8: Analyze missing weather observations

missing = observations["wind_speed_10m"].isna()

missing_inside = missing & inside
missing_outside = missing & ~inside

print("Total missing:", missing.sum())

print("Missing inside ERA5 coverage:", missing_inside.sum())

print("Missing outside ERA5 coverage:", missing_outside.sum())

print("\nERA5 time range:")
print("Start:", surface.time.min().values)
print("End:", surface.time.max().values)

print("\nMissing observations time range:")
print("Start:", observations.loc[missing, "timestamp_utc"].min())
print("End:", observations.loc[missing, "timestamp_utc"].max())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Geographic coverage
axes[0].bar(
    ["Inside ERA5\ncoverage", "Outside ERA5\ncoverage"],
    [599089, 22991]
)

axes[0].set_title("Geographic Coverage")
axes[0].set_ylabel("Aircraft Observations")

axes[0].text(0, 599089, "599,089", ha="center", va="bottom")
axes[0].text(1, 22991, "22,991", ha="center", va="bottom")

# 2. Time coverage
axes[1].bar(
    ["Available hours\n00:00–23:00", "After 23:00"],
    [598193, 23887]
)

axes[1].set_title("Time Coverage")
axes[1].set_ylabel("Aircraft Observations")

axes[1].text(0, 598193, "598,193", ha="center", va="bottom")
axes[1].text(1, 23887, "23,887", ha="center", va="bottom")

plt.suptitle("Aircraft Data Coverage Analysis", fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Step 9: Inspect pressure-level weather data

print("Pressure dataset:")
print(pressure)

print("\nAvailable variables:")
print(list(pressure.data_vars))

print("\nDimensions:")
print(pressure.sizes)

print("\nPressure levels:")
print(pressure["pressure_level"].values)

In [ ]:
# Step 10: Prepare pressure-level weather data

# Sort latitude in ascending order
pressure_sorted = pressure.sortby("latitude")

# Select pressure-level weather variables
pressure_weather_variables = [
    "u_component_of_wind",
    "v_component_of_wind",
    "wind_speed",
    "temperature_C",
    "geopotential_height_m"
]

print("Pressure dataset prepared successfully!")

print("\nVariables:")
print(list(pressure_sorted.data_vars))

print("\nPressure levels:")
print(pressure_sorted["pressure_level"].values)

print("\nDimensions:")
print(pressure_sorted.sizes)

In [ ]:
# Step 11: Estimate aircraft pressure from barometric altitude

T0 = 288.15       # Sea-level standard temperature (K)
P0 = 1013.25      # Sea-level standard pressure (hPa)
L = 0.0065        # Temperature lapse rate (K/m)
R = 287.05        # Specific gas constant (J/kg/K)
g = 9.80665       # Gravitational acceleration (m/s²)

# Convert barometric altitude from feet to meters
altitude_m = observations["barometric_altitude_ft"] * 0.3048

# Estimate atmospheric pressure
observations["estimated_pressure_hPa"] = P0 * (
    1 - (L * altitude_m / T0)
) ** (g / (R * L))

print("Aircraft pressure estimation completed!")

print("\nEstimated pressure statistics:")
print(observations["estimated_pressure_hPa"].describe())

In [ ]:
# Step 12: Check pressure-level coverage

min_pressure = pressure_sorted["pressure_level"].min().item()
max_pressure = pressure_sorted["pressure_level"].max().item()

valid_pressure = observations["estimated_pressure_hPa"].notna()

inside_pressure = (
    valid_pressure &
    observations["estimated_pressure_hPa"].between(
        min_pressure, max_pressure
    )
)

print("ERA5 pressure range:", min_pressure, "to", max_pressure, "hPa")

print("Valid aircraft pressure readings:", valid_pressure.sum())

print("Inside ERA5 pressure range:", inside_pressure.sum())

print("Outside ERA5 pressure range:",
      (valid_pressure & ~inside_pressure).sum())

print("Pressure coverage:",
      round(inside_pressure.sum() / valid_pressure.sum() * 100, 2), "%")

In [ ]:
# Step 13: Prepare pressure-level weather columns

pressure_weather_columns = [
    "wind_u_pressure",
    "wind_v_pressure",
    "wind_speed_pressure",
    "temperature_pressure_C",
    "geopotential_height_m"
]

# Initialize columns with missing values
for col in pressure_weather_columns:
    observations[col] = np.nan

print("Pressure weather columns initialized!")
print(pressure_weather_columns)

In [ ]:
# Check pressure matching inputs

print("Observation columns:")
print([
    "timestamp_utc",
    "latitude_deg",
    "longitude_deg",
    "estimated_pressure_hPa"
])

print("\nPressure dimensions:")
print(pressure_sorted.sizes)

print("\nPressure coordinates:")
print(list(pressure_sorted.coords))

In [ ]:
# Step 14: Match pressure-level weather with aircraft observations

# Prepare pressure dataset
pressure_sorted = pressure_sorted.sortby("latitude")

# Process observations in chunks
chunk_size = 50000

for start in range(0, len(observations), chunk_size):

    end = min(start + chunk_size, len(observations))

    chunk = observations.iloc[start:end]

    # Select observations with valid pressure
    valid = (
        chunk["estimated_pressure_hPa"].between(175, 1000) &
        chunk["latitude_deg"].between(20, 23) &
        chunk["longitude_deg"].between(38, 41)
    )

    if not valid.any():
        continue

    selected = chunk.loc[valid]

    # Prepare interpolation coordinates
    times = (
        selected["timestamp_utc"]
        .dt.tz_localize(None)
        .to_numpy()
    )

    pressures = selected["estimated_pressure_hPa"].to_numpy()

    latitudes = selected["latitude_deg"].to_numpy()

    longitudes = selected["longitude_deg"].to_numpy()

    # Interpolate weather variables
    weather = pressure_sorted[
        pressure_weather_variables
    ].interp(
        valid_time=xr.DataArray(times, dims="obs"),
        pressure_level=xr.DataArray(pressures, dims="obs"),
        latitude=xr.DataArray(latitudes, dims="obs"),
        longitude=xr.DataArray(longitudes, dims="obs"),
        method="linear"
    )

    # Save matched weather values
    observations.loc[selected.index, pressure_weather_columns] = np.column_stack([
        weather["u_component_of_wind"].values,
        weather["v_component_of_wind"].values,
        weather["wind_speed"].values,
        weather["temperature_C"].values,
        weather["geopotential_height_m"].values
    ])

    print(f"Processed {end:,} / {len(observations):,}")

print("\nPressure-level weather matching completed!")

In [ ]:
# Step 15: Check pressure weather matching results

print("Total observations:", len(observations))

print("Matched observations:",
      observations["wind_speed_pressure"].notna().sum())

print("Missing observations:",
      observations["wind_speed_pressure"].isna().sum())

print("Coverage percentage:",
      round(
          observations["wind_speed_pressure"].notna().mean() * 100, 2
      ), "%")

print("\nMissing values by pressure variable:")
print(observations[pressure_weather_columns].isna().sum())

In [ ]:
# Step 16: Analyze missing pressure weather data

pressure_missing = observations["wind_speed_pressure"].isna()

missing_pressure = (
    ~observations["estimated_pressure_hPa"].between(175, 1000)
)

missing_location = (
    ~observations["latitude_deg"].between(20, 23) |
    ~observations["longitude_deg"].between(38, 41)
)

missing_time = (
    observations["timestamp_utc"].dt.tz_localize(None) >
    pressure_sorted["valid_time"].max().values
)

print("Total missing:", pressure_missing.sum())

print("Outside pressure range:",
      (pressure_missing & missing_pressure).sum())

print("Outside geographic coverage:",
      (pressure_missing & missing_location).sum())

print("Outside time range:",
      (pressure_missing & missing_time).sum())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Chart 1: Overall matching results
labels = ["Matched", "Missing"]
values = [530488, 91592]
colors = ["#2E8B57", "#D97706"]

axes[0].bar(labels, values, color=colors, width=0.55)

axes[0].set_title("Overall Pressure Weather Matching")
axes[0].set_ylabel("Number of Observations")
axes[0].set_ylim(0, 650000)

for i, value in enumerate(values):
    percentage = value / 622080 * 100
    axes[0].text(
        i, value + 10000,
        f"{value:,}\n({percentage:.2f}%)",
        ha="center"
    )

# Chart 2: Missing data reasons
reasons = [
    "Pressure range",
    "Geographic coverage",
    "Time coverage"
]

counts = [46852, 22991, 24941]

axes[1].barh(reasons, counts, color="#64748B")
axes[1].set_title("Reasons for Missing Observations")
axes[1].set_xlabel("Number of Observations")

for i, value in enumerate(counts):
    axes[1].text(value + 500, i, f"{value:,}", va="center")

axes[1].set_xlim(0, 55000)

fig.suptitle("Pressure-Level Weather Matching Analysis", fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Step 18: Pressure-level weather summary

# Select observations with matched pressure weather
pressure_matched = observations[
    observations["wind_speed_pressure"].notna()
]

print("Total observations:", len(observations))
print("Matched pressure observations:", len(pressure_matched))

print("\nMissing pressure weather values:")
print(observations[pressure_weather_columns].isna().sum())

print("\nPressure-level weather statistics:")
display(pressure_matched[pressure_weather_columns].describe().round(2))

In [ ]:
# Step 19: Save observations with weather data

observations.to_parquet(
    "observations_with_weather.parquet",
    index=False
)

print("Saved successfully!")
print("Rows:", len(observations))
print("Columns:", len(observations.columns))

In [ ]:
# Get unique aircraft from flights

aircraft_list = (
    flights[["icao24"]]
    .drop_duplicates()
    .copy()
)

aircraft_list["icao24"] = (
    aircraft_list["icao24"].astype(str).str.strip().str.upper()
)

print("Total flights:", len(flights))
print("Unique aircraft:", len(aircraft_list))

display(aircraft_list.head())

In [ ]:
aircraft_db = pd.read_sql_query(
    "SELECT * FROM aircrafts",
    conn
)

print("Total records:", len(aircraft_db))
print("Columns:", aircraft_db.columns.tolist())

display(aircraft_db.head())

In [ ]:
aircraft_db["aircraft_icao_address"] = (
    aircraft_db["aircraft_icao_address"]
    .astype(str).str.strip().str.upper()
)

aircraft_info = aircraft_list.merge(
    aircraft_db,
    left_on="icao24",
    right_on="aircraft_icao_address",
    how="left"
)

aircraft_info = aircraft_info.rename(columns={
    "aircraft_type_code": "typecode",
    "aircraft_registration": "registration",
    "aircraft_manufacturer_icao": "manufacturer",
    "aircraft_operator_icao": "operator"
})

print("Total aircraft:", len(aircraft_info))
print("Matched:", aircraft_info["typecode"].notna().sum())
print("Unmatched:", aircraft_info["typecode"].isna().sum())

display(aircraft_info.head())

In [ ]:
from openap import prop

aircraft_specs = []

for ac in aircraft_info["typecode"].dropna().unique():
    try:
        specs = prop.aircraft(ac, use_synonym=True)

        aircraft_specs.append({
            "typecode": ac,
            "aircraft_name": specs["aircraft"],
            "OEW_kg": specs["oew"],
            "MTOW_kg": specs["mtow"],
            "MLW_kg": specs["mlw"],
            "MFC_kg": specs["mfc"]
        })

    except Exception:
        continue

aircraft_specs = pd.DataFrame(aircraft_specs)

print("Aircraft specifications:", len(aircraft_specs))
display(aircraft_specs)

In [ ]:
aircraft_info = aircraft_info.merge(
    aircraft_specs,
    on="typecode",
    how="left"
)

print("Total aircraft:", len(aircraft_info))
print("With OpenAP specifications:",
      aircraft_info["OEW_kg"].notna().sum())
print("Without specifications:",
      aircraft_info["OEW_kg"].isna().sum())

display(aircraft_info.head())

In [ ]:
# Normalize ICAO24 identifiers
flights["icao24"] = flights["icao24"].str.upper().str.strip()
aircraft_info["icao24"] = aircraft_info["icao24"].str.upper().str.strip()

# Merge flight data with aircraft information
flight_data = flights.merge(
    aircraft_info,
    on="icao24",
    how="left"
)

print("Total flights:", len(flight_data))
print("Flights with aircraft specifications:",
      flight_data["OEW_kg"].notna().sum())
print("Flights without specifications:",
      flight_data["OEW_kg"].isna().sum())

display(flight_data.head())

In [ ]:
departure_data = flight_data[
    flight_data["runway_status"] == "accepted_departure_candidate"
].copy()

fuel_candidate_data = departure_data[
    departure_data["OEW_kg"].notna()
].copy()

print("Departure candidates:", len(departure_data))
print("With OpenAP specifications:", len(fuel_candidate_data))
print("Without specifications:",
      departure_data["OEW_kg"].isna().sum())

display(fuel_candidate_data.head())

In [ ]:
fuel_candidate_data = fuel_candidate_data.copy()

fuel_candidate_data["mass_80_kg"] = (
    fuel_candidate_data["MTOW_kg"] * 0.80
)

display(
    fuel_candidate_data[
        ["flight_id", "icao24", "typecode", "MTOW_kg", "mass_80_kg"]
    ].head(10)
)

In [ ]:
# Step 28: Select one sample flight

sample_flight = fuel_candidate_data.iloc[0]

print("Flight ID:", sample_flight["flight_id"])
print("Aircraft:", sample_flight["typecode"])
print("ICAO24:", sample_flight["icao24"])
print("Estimated Mass (kg):", sample_flight["mass_80_kg"])

In [ ]:
# Step 29: Get observations for the selected flight

flight_observations = observations[
    (observations["icao24"].astype(str).str.upper()
     == str(sample_flight["icao24"]).upper()) &
    (observations["timestamp_ms"] >= sample_flight["anchor_ms"]) &
    (observations["timestamp_ms"] <= sample_flight["end_ms"])
].copy()

flight_observations = flight_observations.sort_values(
    "timestamp_ms"
).reset_index(drop=True)

print("Flight observations:", len(flight_observations))
display(flight_observations.head())

In [ ]:
# Step 30: Check observation data

columns_to_check = [
    "timestamp_ms",
    "barometric_altitude_ft",
    "barometric_rate_fpm",
    "ground_speed_kt",
    "track_deg"
]

display(flight_observations[columns_to_check].describe())

In [ ]:
# Step 30: Filter departure observations

departure_observations = flight_observations[
    flight_observations["barometric_altitude_ft"] <= 10000
].copy()

departure_observations = departure_observations.sort_values(
    "timestamp_ms"
).reset_index(drop=True)

print("Departure observations:", len(departure_observations))
print("Minimum altitude:",
      departure_observations["barometric_altitude_ft"].min())
print("Maximum altitude:",
      departure_observations["barometric_altitude_ft"].max())

In [ ]:
# Step 31: Estimate True Airspeed

wind_u = departure_observations["wind_u_pressure"].fillna(
    departure_observations["wind_u_10m"]
)

wind_v = departure_observations["wind_v_pressure"].fillna(
    departure_observations["wind_v_10m"]
)

track_rad = np.deg2rad(departure_observations["track_deg"])

along_track_wind = (
    wind_u * np.sin(track_rad) +
    wind_v * np.cos(track_rad)
)

departure_observations["TAS_kt"] = (
    departure_observations["ground_speed_kt"]
    - along_track_wind * 1.94384
)

display(
    departure_observations[
        ["ground_speed_kt", "TAS_kt"]
    ].describe()
)

In [ ]:
# Step 32: Check missing values

required_columns = [
    "timestamp_ms",
    "barometric_altitude_ft",
    "barometric_rate_fpm",
    "TAS_kt"
]

print(departure_observations[required_columns].isna().sum())

In [ ]:
#OpenAP FuelFlow model

fuel_model = FuelFlow(ac=sample_flight["typecode"])

print("Aircraft:", sample_flight["typecode"])
print("Mass:", sample_flight["mass_80_kg"], "kg")

In [ ]:
# Step 34: Calculate fuel flow

departure_observations["fuel_flow_kg_s"] = fuel_model.enroute(
    mass=sample_flight["mass_80_kg"],
    tas=departure_observations["TAS_kt"],
    alt=departure_observations["barometric_altitude_ft"],
    vs=departure_observations["barometric_rate_fpm"]
)

display(
    departure_observations[
        ["barometric_altitude_ft", "TAS_kt",
         "barometric_rate_fpm", "fuel_flow_kg_s"]
    ].head(10)
)

In [ ]:
# Step 35: Fuel flow statistics

print(departure_observations["fuel_flow_kg_s"].describe())

print(
    "Missing fuel flow:",
    departure_observations["fuel_flow_kg_s"].isna().sum()
)

print(
    "Non-positive fuel flow:",
    (departure_observations["fuel_flow_kg_s"] <= 0).sum()
)

In [ ]:
# Step 36: Calculate time intervals

departure_observations["time_seconds"] = (
    departure_observations["timestamp_ms"] -
    departure_observations["timestamp_ms"].iloc[0]
) / 1000

print("Duration (minutes):",
      departure_observations["time_seconds"].iloc[-1] / 60)

print("First intervals:")
display(departure_observations[
    ["timestamp_utc", "time_seconds", "fuel_flow_kg_s"]
].head())

In [ ]:
# Step 37: Calculate total departure fuel

total_departure_fuel_kg = np.trapezoid(
    departure_observations["fuel_flow_kg_s"],
    x=departure_observations["time_seconds"]
)

print("Total Departure Fuel:",
      round(total_departure_fuel_kg, 2), "kg")

In [ ]:
# Step 39: Check FuelFlow support

from openap import FuelFlow

aircraft_types = fuel_candidate_data["typecode"].dropna().unique()

supported_aircraft = []
unsupported_aircraft = []

for ac in aircraft_types:
    try:
        FuelFlow(ac=ac)
        supported_aircraft.append(ac)
    except Exception as e:
        unsupported_aircraft.append({
            "typecode": ac,
            "error": str(e)
        })

print("Supported:", len(supported_aircraft))
print("Unsupported:", len(unsupported_aircraft))

print("\nSupported aircraft:")
print(supported_aircraft)

print("\nUnsupported aircraft:")
display(pd.DataFrame(unsupported_aircraft))

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module=r"openap\.(drag|prop)"
)


In [ ]:
# Step 40: Test alternative aircraft names

from openap import FuelFlow

for ac in ["A21N", "B77L", "B77W"]:
    try:
        model = FuelFlow(ac=ac)
        print(ac, "-> Supported")
    except Exception as e:
        print(ac, "-> Unsupported:", str(e))

In [ ]:
# Step 42: Test FuelFlow with synonyms

for ac in ["A21N", "B77L"]:
    try:
        model = FuelFlow(ac=ac, use_synonym=True)
        print(ac, "-> Supported with synonym")
    except Exception as e:
        print(ac, "-> Still unsupported:", str(e))

In [ ]:
# Step 43: Check all aircraft using synonyms
# Suppress OpenAP UserWarnings

from openap import FuelFlow

supported_aircraft = []
unsupported_aircraft = []

for ac in fuel_candidate_data["typecode"].dropna().unique():
    try:
        FuelFlow(ac=ac, use_synonym=True)
        supported_aircraft.append(ac)
    except Exception as e:
        unsupported_aircraft.append({
            "typecode": ac,
            "error": str(e)
        })

print("Supported:", len(supported_aircraft))
print(supported_aircraft)

print("Unsupported:", len(unsupported_aircraft))
display(pd.DataFrame(unsupported_aircraft))

In [ ]:
supported_fuel_data = fuel_candidate_data[
    fuel_candidate_data["typecode"].isin(supported_aircraft)
].copy()

unsupported_fuel_data = fuel_candidate_data[
    ~fuel_candidate_data["typecode"].isin(supported_aircraft)
].copy()

print("Total fuel candidates:", len(fuel_candidate_data))
print("Supported flights:", len(supported_fuel_data))
print("Unsupported flights:", len(unsupported_fuel_data))

In [ ]:
# Step 45: Define departure fuel calculation

def calculate_departure_fuel(flight, observations):

    ac = flight["typecode"]
    mass = flight["mass_80_kg"]

    obs = observations[
        (observations["icao24"].astype(str).str.upper()
         == str(flight["icao24"]).upper()) &
        (observations["timestamp_ms"] >= flight["anchor_ms"]) &
        (observations["timestamp_ms"] <= flight["end_ms"])
    ].copy()

    obs = obs.sort_values("timestamp_ms")

    obs = obs[
        obs["barometric_altitude_ft"] <= 10000
    ].copy()

    if len(obs) < 2:
        raise ValueError("Insufficient departure observations")

    wind_u = obs["wind_u_pressure"].fillna(obs["wind_u_10m"])
    wind_v = obs["wind_v_pressure"].fillna(obs["wind_v_10m"])

    track_rad = np.deg2rad(obs["track_deg"])

    along_track_wind = (
        wind_u * np.sin(track_rad) +
        wind_v * np.cos(track_rad)
    )

    obs["TAS_kt"] = (
        obs["ground_speed_kt"] - along_track_wind * 1.94384
    )

    obs = obs.dropna(subset=[
        "timestamp_ms",
        "barometric_altitude_ft",
        "barometric_rate_fpm",
        "TAS_kt"
    ])

    obs = obs[obs["TAS_kt"] > 0].copy()

    if len(obs) < 2:
        raise ValueError("Insufficient valid observations")

    model = FuelFlow(ac=ac, use_synonym=True)

    obs["fuel_flow_kg_s"] = model.enroute(
        mass=mass,
        tas=obs["TAS_kt"],
        alt=obs["barometric_altitude_ft"],
        vs=obs["barometric_rate_fpm"]
    )

    obs["time_seconds"] = (
        obs["timestamp_ms"] - obs["timestamp_ms"].iloc[0]
    ) / 1000

    total_fuel = np.trapezoid(
        obs["fuel_flow_kg_s"],
        x=obs["time_seconds"]
    )

    return {
        "flight_id": flight["flight_id"],
        "icao24": flight["icao24"],
        "aircraft_type": ac,
        "estimated_mass_kg": mass,
        "departure_points": len(obs),
        "departure_duration_min": (
            obs["time_seconds"].iloc[-1] / 60
        ),
        "total_departure_fuel_kg": total_fuel
    }

In [ ]:
# Step 46: Calculate fuel for all supported flights

flight_results = []
failed_flights = []

for _, flight in supported_fuel_data.iterrows():

    try:
        result = calculate_departure_fuel(
            flight,
            observations
        )
        flight_results.append(result)

    except Exception as e:
        failed_flights.append({
            "flight_id": flight["flight_id"],
            "typecode": flight["typecode"],
            "error": str(e)
        })

fuel_results = pd.DataFrame(flight_results)
failed_flights_df = pd.DataFrame(failed_flights)

print("Successfully calculated:", len(fuel_results))
print("Failed flights:", len(failed_flights_df))

display(fuel_results.head())

In [ ]:
# Step 47: Investigate failed flights

print("Failed flights:", len(failed_flights_df))

display(
    failed_flights_df["error"]
    .value_counts()
    .reset_index(name="count")
)

In [ ]:
# Keep successfully calculated flights only

final_fuel_results = fuel_results.copy()

print("Total original flights:", len(fuel_candidate_data))
print("Successfully calculated:", len(final_fuel_results))
print("Excluded flights:", len(failed_flights_df))
print("Final dataset:", len(final_fuel_results))

display(final_fuel_results.head())

In [ ]:
# Step 56: Save final fuel results

final_fuel_results.to_csv(
    "OEJN_departure_fuel_results.csv",
    index=False
)

final_fuel_results.to_parquet(
    "OEJN_departure_fuel_results.parquet",
    index=False
)
print("Files saved successfully!")

In [ ]:
print(final_fuel_results.columns.tolist())

In [ ]:

from openap import FuelFlow
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=UserWarning)


def calculate_fuel_points(flight, observations):

    ac = flight["typecode"]
    mass = flight["mass_80_kg"]

    obs = observations[
        (observations["icao24"].astype(str).str.upper()
         == str(flight["icao24"]).upper()) &
        (observations["timestamp_ms"] >= flight["anchor_ms"]) &
        (observations["timestamp_ms"] <= flight["end_ms"])
    ].copy()

    obs = obs.sort_values("timestamp_ms")
    obs = obs[obs["barometric_altitude_ft"] <= 10000].copy()

    if len(obs) < 2:
        return None

    # Original TAS calculation
    wind_u = obs["wind_u_pressure"].fillna(obs["wind_u_10m"])
    wind_v = obs["wind_v_pressure"].fillna(obs["wind_v_10m"])

    track_rad = np.deg2rad(obs["track_deg"])

    along_track_wind = (
        wind_u * np.sin(track_rad) +
        wind_v * np.cos(track_rad)
    )

    obs["TAS_kt"] = (
        obs["ground_speed_kt"] - along_track_wind * 1.94384
    )

    obs = obs.dropna(subset=[
        "timestamp_ms",
        "barometric_altitude_ft",
        "barometric_rate_fpm",
        "TAS_kt"
    ])

    obs = obs[obs["TAS_kt"] > 0].copy()

    if len(obs) < 2:
        return None

    model = FuelFlow(ac=ac, use_synonym=True)

    obs["fuel_flow_kg_s"] = model.enroute(
        mass=mass,
        tas=obs["TAS_kt"],
        alt=obs["barometric_altitude_ft"],
        vs=obs["barometric_rate_fpm"]
    )

    obs["flight_id"] = flight["flight_id"]
    obs["aircraft_type"] = ac
    obs["estimated_mass_kg"] = mass

    return obs


# Select successful flights
successful_flights = supported_fuel_data[
    supported_fuel_data["flight_id"].isin(fuel_results["flight_id"])
]

# Calculate fuel flow for each flight
fuel_points_list = []

for _, flight in tqdm(
    successful_flights.iterrows(),
    total=len(successful_flights)
):
    points = calculate_fuel_points(flight, observations)

    if points is not None:
        fuel_points_list.append(points)

if not fuel_points_list:
    raise ValueError("No fuel-flow data was generated.")


# Combine all observations
weather_fuel_dataset = pd.concat(
    fuel_points_list,
    ignore_index=True
)


# Add total departure fuel consumption
total_fuel = final_fuel_results[
    ["flight_id", "total_departure_fuel_kg"]
].drop_duplicates(subset=["flight_id"])

weather_fuel_dataset = weather_fuel_dataset.merge(
    total_fuel,
    on="flight_id",
    how="left"
)


# Select important columns
selected_columns = [
    "flight_id",
    "icao24",
    "aircraft_type",
    "timestamp_utc",
    "timestamp_ms",
    "latitude",
    "longitude",
    "estimated_mass_kg",
    "barometric_altitude_ft",
    "barometric_rate_fpm",
    "ground_speed_kt",
    "track_deg",
    "TAS_kt",
    "wind_u_pressure",
    "wind_v_pressure",
    "wind_u_10m",
    "wind_v_10m",
    "temperature_2m_C",
    "wind_speed_10m",
    "surface_pressure_hPa",
    "fuel_flow_kg_s",
    "total_departure_fuel_kg"
]

# Keep available columns
selected_columns = [
    col for col in selected_columns
    if col in weather_fuel_dataset.columns
]

weather_fuel_dataset = weather_fuel_dataset[selected_columns].copy()


# Sort by flight and timestamp
weather_fuel_dataset = weather_fuel_dataset.sort_values(
    ["flight_id", "timestamp_ms"]
).reset_index(drop=True)


# Save the dataset
weather_fuel_dataset.to_csv(
    "Weather_FuelFlow_Dataset.csv",
    index=False
)

weather_fuel_dataset.to_parquet(
    "Weather_FuelFlow_Dataset.parquet",
    index=False
)


# Display dataset summary
print("Dataset saved successfully!")
print("Total observations:", len(weather_fuel_dataset))
print("Unique flights:", weather_fuel_dataset["flight_id"].nunique())
print("Total features:", len(weather_fuel_dataset.columns))
print("Missing Fuel Flow:", weather_fuel_dataset["fuel_flow_kg_s"].isna().sum())
print("Missing Total Departure Fuel:",
      weather_fuel_dataset["total_departure_fuel_kg"].isna().sum())

print("\nDataset columns:")
print(weather_fuel_dataset.columns.tolist())

display(weather_fuel_dataset.head())


In [ ]:
from google.colab import files

files.download("Weather_FuelFlow_Dataset.csv")